In [1]:
from pathlib import Path
import pandas as pd

# Resolve paths from the project root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "datasets" / "FinScope2024Rwanda.dta"
OUTPUT_PATH = ROOT / "data" / "processed" / "population_group_indicators.csv"

columns = [
    "b1",
    "b2",
    "c4a",
    "a6",
    "pop_wt",
    "k1",
    "f5",
    "h2_1",
    "g3a_17",
    "e9b",
    "g3a_11",
]

df = pd.read_stata(DATA_PATH, columns=columns, convert_categoricals=True)

# Define the six binary indicators.
df["bank_account"] = df["k1"].eq("Yes").astype(int)

df["mobile_money_registered"] = (
    df["f5"].eq("You are a registered mobile money user")
).astype(int)

df["borrowed_recently"] = df["h2_1"].eq("Yes").astype(int)
df["does_not_save"] = df["g3a_17"].eq("Yes").astype(int)
df["emergency_funds"] = df["e9b"].eq("Yes").astype(int)
df["group_saving"] = df["g3a_11"].eq("Yes").astype(int)

# Create age groups consistent with the earlier analysis.
df["Age group"] = pd.cut(
    df["b1"],
    bins=[15, 24, 34, 44, 54, 64, float("inf")],
    labels=["16–24", "25–34", "35–44", "45–54", "55–64", "65+"],
)

# Standardize group names for the output.
df["Gender"] = df["b2"]
df["Education"] = df["c4a"]
df["Area"] = df["a6"]

group_columns = ["Gender", "Age group", "Education", "Area"]

indicator_columns = {
    "Owns a bank account (%)": "bank_account",
    "Registered mobile-money user (%)": "mobile_money_registered",
    "Borrowed in the past 12 months (%)": "borrowed_recently",
    "Does not save (%)": "does_not_save",
    "Has emergency funds (%)": "emergency_funds",
    "Saves with a group (%)": "group_saving",
}

results = []

for group_type in group_columns:
    for group_name, subset in df.groupby(group_type, observed=True, dropna=True):
        # Retain respondents with valid positive survey weights.
        valid_weight = subset["pop_wt"].notna() & (subset["pop_wt"] > 0)
        subset = subset.loc[valid_weight]

        if subset.empty:
            continue

        row = {
            "Group type": group_type,
            "Group": str(group_name),
        }

        for output_name, source_column in indicator_columns.items():
            # Exclude respondents whose source answer is missing.
            valid = subset[source_column].notna()
            data = subset.loc[valid]

            row.setdefault("Valid respondents", len(subset))

            denominator = data["pop_wt"].sum()
            estimate = (
                (data[source_column] * data["pop_wt"]).sum() / denominator * 100
                if denominator > 0
                else float("nan")
            )

            row[output_name] = estimate

        results.append(row)

population_groups = pd.DataFrame(results)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
population_groups.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(population_groups))
print("Columns:", population_groups.columns.tolist())
display(population_groups.head(15))

Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/population_group_indicators.csv
Rows: 17
Columns: ['Group type', 'Group', 'Valid respondents', 'Owns a bank account (%)', 'Registered mobile-money user (%)', 'Borrowed in the past 12 months (%)', 'Does not save (%)', 'Has emergency funds (%)', 'Saves with a group (%)']


,Group type,Group,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%),Has emergency funds (%),Saves with a group (%)
0,Gender,Male,6186,25.316027,72.158394,39.204948,18.719602,22.093775,49.136665
1,Gender,Female,7808,15.483023,61.496916,39.794598,21.655717,19.561064,54.306069
2,Age group,16–24,1777,14.371340,58.518479,27.508041,32.751138,17.066433,38.220344
3,Age group,25–34,2900,24.324205,77.812175,46.932949,12.446100,22.236124,60.182749
4,Age group,35–44,3788,24.774861,73.476228,46.164163,12.232948,23.632289,58.835234
5,Age group,45–54,2180,22.824941,70.540572,45.683314,13.291921,22.706954,59.207092
6,Age group,55–64,1548,18.974636,59.925160,43.385341,20.395095,21.220838,54.315478
7,Age group,65+,1801,14.608214,45.565647,30.341547,28.943939,19.150845,44.846252
8,Education,No formal education,2510,6.039635,41.372298,32.475230,26.888432,13.873574,52.579034
9,Education,Primary 1-3,1875,10.198676,55.268440,39.474609,20.858125,16.739005,55.922784


In [2]:

from pathlib import Path
import pandas as pd

# Resolve paths from the project root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "datasets" / "FinScope2024Rwanda.dta"
OUTPUT_PATH = ROOT / "data" / "processed" / "population_group_indicators.csv"

columns = ["b1", "b2", "c4a", "a6", "pop_wt", "k1", "f5", "h2_1", "g3a_17"]

df = pd.read_stata(
    DATA_PATH,
    columns=columns,
    convert_categoricals=True
)

# Define the four binary indicators.
df["bank_account"] = df["k1"].eq("Yes").astype(int)

df["mobile_money_registered"] = (
    df["f5"].eq("You are a registered mobile money user")
).astype(int)

df["borrowed_recently"] = df["h2_1"].eq("Yes").astype(int)
df["does_not_save"] = df["g3a_17"].eq("Yes").astype(int)

# Create age groups consistent with the earlier analysis.
df["Age group"] = pd.cut(
    df["b1"],
    bins=[15, 24, 34, 44, 54, 64, float("inf")],
    labels=["16–24", "25–34", "35–44", "45–54", "55–64", "65+"]
)

# Standardize group names for the output.
df["Gender"] = df["b2"]
df["Education"] = df["c4a"]
df["Area"] = df["a6"]

group_columns = ["Gender", "Age group", "Education", "Area"]

indicator_columns = {
    "Owns a bank account (%)": "bank_account",
    "Registered mobile-money user (%)": "mobile_money_registered",
    "Borrowed in the past 12 months (%)": "borrowed_recently",
    "Does not save (%)": "does_not_save",
}

results = []

for group_type in group_columns:
    for group_name, subset in df.groupby(group_type, observed=True, dropna=True):
        valid_weight = subset["pop_wt"].notna() & (subset["pop_wt"] > 0)
        subset = subset.loc[valid_weight]

        if subset.empty:
            continue

        row = {
            "Group type": group_type,
            "Group": str(group_name),
            "Valid respondents": len(subset),
        }

        for output_name, source_column in indicator_columns.items():
            valid = subset[source_column].notna()
            data = subset.loc[valid]

            denominator = data["pop_wt"].sum()
            estimate = (
                (data[source_column] * data["pop_wt"]).sum() / denominator * 100
                if denominator > 0
                else float("nan")
            )

            row[output_name] = estimate

        results.append(row)

population_groups = pd.DataFrame(results)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
population_groups.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(population_groups))
display(population_groups.head(15))


Saved: /home/ibrahim/Desktop/Backup/PPioneers/nisr-big-data-hackathon-project/data/processed/population_group_indicators.csv
Rows: 17


,Group type,Group,Valid respondents,Owns a bank account (%),Registered mobile-money user (%),Borrowed in the past 12 months (%),Does not save (%)
0,Gender,Male,6186,25.316027,72.158394,39.204948,18.719602
1,Gender,Female,7808,15.483023,61.496916,39.794598,21.655717
2,Age group,16–24,1777,14.371340,58.518479,27.508041,32.751138
3,Age group,25–34,2900,24.324205,77.812175,46.932949,12.446100
4,Age group,35–44,3788,24.774861,73.476228,46.164163,12.232948
5,Age group,45–54,2180,22.824941,70.540572,45.683314,13.291921
6,Age group,55–64,1548,18.974636,59.925160,43.385341,20.395095
7,Age group,65+,1801,14.608214,45.565647,30.341547,28.943939
8,Education,No formal education,2510,6.039635,41.372298,32.475230,26.888432
9,Education,Primary 1-3,1875,10.198676,55.268440,39.474609,20.858125
